<a href="https://colab.research.google.com/github/EMR6040/CNN-PROJECT/blob/main/GAN_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

model.save('mnist_model.h5')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 54s 28ms/step - accuracy: 0.9580 - loss: 0.1433 - val_accuracy: 0.9836 - val_loss: 0.0541
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 28ms/step - accuracy: 0.9855 - loss: 0.0480 - val_accuracy: 0.9891 - val_loss: 0.0340
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 26ms/step - accuracy: 0.9899 - loss: 0.0327 - val_accuracy: 0.9916 - val_loss: 0.0295
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.9916 - loss: 0.0253 - val_accuracy: 0.9888 - val_loss: 0.0356
Epoch 5/5
  89/1875 ━━━━━━━━━━━━━━━━━━━━ 40s 23ms/step - accuracy: 0.9956 - loss: 0.0115

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
import cv2

model = tf.keras.models.load_model('mnist_model.h5')

def predict_digit(data):
    if isinstance(data, dict):
        img = data['composite']
    else:
        img = data

    if img is None:
        return "Lütfen bir rakam çizin."

    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    img = cv2.resize(img, (28, 28))

    if np.mean(img) > 127:
        img = cv2.bitwise_not(img)

    img = img / 255.0
    img = img.reshape(1, 28, 28, 1)


    prediction = model.predict(img).flatten()

    return {str(i): float(prediction[i]) for i in range(10)}

with gr.Blocks() as demo:

    gr.Markdown("## MNIST El Yazısı Tanıma")
    with gr.Row():
        input_canvas = gr.Sketchpad(label="Rakam Çiz", type="numpy")
        output_label = gr.Label(num_top_classes=3, label="Tahmin")

    submit_btn = gr.Button("Tahmin Et")
    submit_btn.click(fn=predict_digit, inputs=input_canvas, outputs=output_label)

    gr.ClearButton([input_canvas, output_label])

demo.launch(debug=True, share=True)